In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
BASE_PATH = "/content/drive/MyDrive/Cloud anomaly detection/dataset"

In [ ]:
system_df = pd.read_csv(
    f"{BASE_PATH}/system_metrics/raw/borg_traces_data.csv"
)

In [ ]:
system_df.shape

(405894, 34)

In [ ]:
system_df.columns.tolist()

['Unnamed: 0',
 'time',
 'instance_events_type',
 'collection_id',
 'scheduling_class',
 'collection_type',
 'priority',
 'alloc_collection_id',
 'instance_index',
 'machine_id',
 'resource_request',
 'constraint',
 'collections_events_type',
 'user',
 'collection_name',
 'collection_logical_name',
 'start_after_collection_ids',
 'vertical_scaling',
 'scheduler',
 'start_time',
 'end_time',
 'average_usage',
 'maximum_usage',
 'random_sample_usage',
 'assigned_memory',
 'page_cache_memory',
 'cycles_per_instruction',
 'memory_accesses_per_instruction',
 'sample_rate',
 'cpu_usage_distribution',
 'tail_cpu_usage_distribution',
 'cluster',
 'event',
 'failed']

In [ ]:
system_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 405894 entries, 0 to 405893
Data columns (total 34 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   Unnamed: 0                       405894 non-null  int64  
 1   time                             405894 non-null  int64  
 2   instance_events_type             405894 non-null  int64  
 3   collection_id                    405894 non-null  int64  
 4   scheduling_class                 405894 non-null  int64  
 5   collection_type                  405894 non-null  int64  
 6   priority                         405894 non-null  int64  
 7   alloc_collection_id              405894 non-null  int64  
 8   instance_index                   405894 non-null  int64  
 9   machine_id                       405894 non-null  int64  
 10  resource_request                 405120 non-null  object 
 11  constraint                       405894 non-null  object 
 12  co

In [ ]:
system_df.head()

,Unnamed: 0,time,instance_events_type,collection_id,scheduling_class,collection_type,priority,alloc_collection_id,instance_index,machine_id,...,assigned_memory,page_cache_memory,cycles_per_instruction,memory_accesses_per_instruction,sample_rate,cpu_usage_distribution,tail_cpu_usage_distribution,cluster,event,failed
0,0,0,2,94591244395,3,1,200,0,144,168846390496,...,0.014435,0.000415,NaN,NaN,1.0,[0.00314331 0.00381088 0.00401306 0.00415039 0...,[0.00535583 0.00541687 0.00548553 0.00554657 0...,7,FAIL,1
1,1,2517305308183,2,260697606809,2,0,360,221495397286,335,85515092,...,0.000000,0.000000,NaN,NaN,1.0,[1.23977661e-05 1.23977661e-05 1.23977661e-05 ...,[1.23977661e-05 1.23977661e-05 1.23977661e-05 ...,7,FAIL,1
2,2,195684022913,6,276227177776,2,0,103,0,376,169321752432,...,0.010422,0.000235,0.939919,0.001318,1.0,[0.01344299 0.01809692 0.0201416 0.02246094 0...,[0.02902222 0.02929688 0.0295105 0.0296936 0...,7,SCHEDULE,0
3,3,0,2,10507389885,3,0,200,0,1977,178294817221,...,0.041626,0.000225,1.359102,0.007643,1.0,[0.03704834 0.04125977 0.04290771 0.04425049 0...,[0.05535889 0.05584717 0.05633545 0.05718994 0...,8,FAIL,1
4,4,1810627494172,3,25911621841,2,0,0,0,3907,231364893292,...,0.000272,0.000010,NaN,NaN,1.0,[0. 0. 0. 0. 0...,[0.00041485 0.00041485 0.00041485 0.00041485 0...,2,FINISH,0


In [ ]:
system_df.isnull().sum()

,0
Unnamed: 0,0
time,0
instance_events_type,0
collection_id,0
scheduling_class,0
collection_type,0
priority,0
alloc_collection_id,0
instance_index,0
machine_id,0


In [ ]:
system_df.duplicated().sum()

np.int64(0)

In [ ]:
system_clean = system_df.copy()

In [ ]:
system_clean.drop(
    columns=['Unnamed: 0'],
    inplace=True
)

In [ ]:
system_clean.shape

(405894, 33)

In [ ]:
system_clean['vertical_scaling'] = system_clean['vertical_scaling'].fillna(
    system_clean['vertical_scaling'].median()
)

system_clean['scheduler'] = system_clean['scheduler'].fillna(
    system_clean['scheduler'].median()
)

system_clean['cycles_per_instruction'] = system_clean['cycles_per_instruction'].fillna(
    system_clean['cycles_per_instruction'].median()
)

system_clean['memory_accesses_per_instruction'] = system_clean['memory_accesses_per_instruction'].fillna(
    system_clean['memory_accesses_per_instruction'].median()
)

In [ ]:
system_clean['resource_request'] = system_clean['resource_request'].fillna(
    'Unknown'
)

In [ ]:
system_clean.isnull().sum().sum()

np.int64(0)

In [ ]:
system_clean[['average_usage']].head()

,average_usage
0,"{'cpus': 0.00466156005859375, 'memory': 0.0059..."
1,"{'cpus': 0.0, 'memory': 9.5367431640625e-07}"
2,"{'cpus': 0.024200439453125, 'memory': 0.002788..."
3,"{'cpus': 0.047607421875, 'memory': 0.034423828..."
4,"{'cpus': 0.000270843505859375, 'memory': 7.629..."


In [ ]:
system_clean[['maximum_usage']].head()

,maximum_usage
0,"{'cpus': 0.01190185546875, 'memory': 0.0059356..."
1,"{'cpus': 0.0, 'memory': 9.5367431640625e-07}"
2,"{'cpus': 0.06005859375, 'memory': 0.0028457641..."
3,"{'cpus': 0.13330078125, 'memory': 0.03466796875}"
4,"{'cpus': 0.00041484832763671875, 'memory': 7.6..."


In [ ]:
system_clean[['random_sample_usage']].head()

,random_sample_usage
0,"{'cpus': 0.0043487548828125, 'memory': None}"
1,"{'cpus': 0.0, 'memory': None}"
2,"{'cpus': 0.026458740234375, 'memory': None}"
3,"{'cpus': 0.05084228515625, 'memory': None}"
4,"{'cpus': 0.0003414154052734375, 'memory': None}"


In [ ]:
system_clean[['cpu_usage_distribution']].head()

,cpu_usage_distribution
0,[0.00314331 0.00381088 0.00401306 0.00415039 0...
1,[1.23977661e-05 1.23977661e-05 1.23977661e-05 ...
2,[0.01344299 0.01809692 0.0201416 0.02246094 0...
3,[0.03704834 0.04125977 0.04290771 0.04425049 0...
4,[0. 0. 0. 0. 0...


In [ ]:
system_clean[['tail_cpu_usage_distribution']].head()

,tail_cpu_usage_distribution
0,[0.00535583 0.00541687 0.00548553 0.00554657 0...
1,[1.23977661e-05 1.23977661e-05 1.23977661e-05 ...
2,[0.02902222 0.02929688 0.0295105 0.0296936 0...
3,[0.05535889 0.05584717 0.05633545 0.05718994 0...
4,[0.00041485 0.00041485 0.00041485 0.00041485 0...


In [ ]:
system_clean['failed'].value_counts()

,count
failed,
0,313216
1,92678


In [ ]:
import ast

In [ ]:
system_clean['avg_cpu'] = system_clean['average_usage'].apply(
    lambda x: ast.literal_eval(x)['cpus']
)

system_clean['avg_memory'] = system_clean['average_usage'].apply(
    lambda x: ast.literal_eval(x)['memory']
)

In [ ]:
system_clean['max_cpu'] = system_clean['maximum_usage'].apply(
    lambda x: ast.literal_eval(x)['cpus']
)

system_clean['max_memory'] = system_clean['maximum_usage'].apply(
    lambda x: ast.literal_eval(x)['memory']
)

In [ ]:
system_clean['sample_cpu'] = system_clean['random_sample_usage'].apply(
    lambda x: ast.literal_eval(x)['cpus']
)

In [ ]:
system_clean[
[
'avg_cpu',
'avg_memory',
'max_cpu',
'max_memory',
'sample_cpu'
]
].head()

,avg_cpu,avg_memory,max_cpu,max_memory,sample_cpu
0,0.004662,5.920410e-03,0.011902,5.935669e-03,0.004349
1,0.000000,9.536743e-07,0.000000,9.536743e-07,0.000000
2,0.024200,2.788544e-03,0.060059,2.845764e-03,0.026459
3,0.047607,3.442383e-02,0.133301,3.466797e-02,0.050842
4,0.000271,7.629395e-05,0.000415,7.629395e-05,0.000341


In [ ]:
system_clean[
[
'avg_cpu',
'avg_memory',
'max_cpu',
'max_memory',
'sample_cpu'
]
].describe()

,avg_cpu,avg_memory,max_cpu,max_memory,sample_cpu
count,405894.000000,405894.000000,405894.000000,400967.000000,405894.000000
mean,0.007446,0.005640,0.025352,0.005946,0.007191
std,0.018567,0.016559,0.052681,0.016632,0.019297
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000200,0.000238,0.000794,0.000262,0.000167
50%,0.001049,0.001282,0.005005,0.001444,0.000701
75%,0.007286,0.004257,0.029724,0.004776,0.007042
max,0.538086,0.223877,1.271484,0.224365,0.644531


In [ ]:
system_clean[
[
'avg_cpu',
'avg_memory',
'max_cpu',
'max_memory'
]
].isnull().sum()

,0
avg_cpu,0
avg_memory,0
max_cpu,0
max_memory,4927


In [ ]:
system_clean['max_memory'].isnull().sum()

np.int64(4927)

In [ ]:
system_clean['max_memory'] = system_clean['max_memory'].fillna(
    system_clean['max_memory'].median()
)

In [ ]:
system_clean[
[
'avg_cpu',
'avg_memory',
'max_cpu',
'max_memory',
'sample_cpu'
]
].isnull().sum()

,0
avg_cpu,0
avg_memory,0
max_cpu,0
max_memory,0
sample_cpu,0


In [ ]:
system_features = system_clean[
[
'avg_cpu',
'avg_memory',
'max_cpu',
'max_memory',
'sample_cpu',

'assigned_memory',
'page_cache_memory',

'cycles_per_instruction',
'memory_accesses_per_instruction',

'vertical_scaling',
'scheduler',

'priority',
'scheduling_class',

'failed'
]
]

In [ ]:
system_features.shape

(405894, 14)

In [ ]:
system_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 405894 entries, 0 to 405893
Data columns (total 14 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   avg_cpu                          405894 non-null  float64
 1   avg_memory                       405894 non-null  float64
 2   max_cpu                          405894 non-null  float64
 3   max_memory                       405894 non-null  float64
 4   sample_cpu                       405894 non-null  float64
 5   assigned_memory                  405894 non-null  float64
 6   page_cache_memory                405894 non-null  float64
 7   cycles_per_instruction           405894 non-null  float64
 8   memory_accesses_per_instruction  405894 non-null  float64
 9   vertical_scaling                 405894 non-null  float64
 10  scheduler                        405894 non-null  float64
 11  priority                         405894 non-null  int64  
 12  sc

In [ ]:
system_features.to_csv(
"/content/drive/MyDrive/Cloud anomaly detection/processed/system_features.csv",
index=False
)